# Business Entity Resolution: Kaggle pipeline

Runs prep → blocking → features → training → lockbox accuracy → test submission.
Every step is **skipped if it already finished**, so you can resume after a crash or in a new session.
Full instructions: `KAGGLE.md` in the repo.

**Before the first run:** Settings → Internet **on**, Accelerator **None** (CPU). Add Input → your private
dataset with the challenge data. To resume a previous run, also add that run's notebook output as input.

## 1. Settings: change these for each experiment

In [ ]:
REPO_URL   = "https://github.com/saiboddapati2k3/amz-ml-challenge.git"
BRANCH     = "main"

BLOCK_TAG  = "v3h1"        # new name after changing src/prep.py, src/normalize.py or src/block.py
MODEL_TAG  = "dev_frozen"  # model_<MODEL_TAG>_*.txt; dev_frozen is committed in the repo
TRAIN_MODEL = False        # True: train MODEL_TAG on the dev features of BLOCK_TAG
TAU        = 0.75          # decision threshold

RUN_DEV     = True         # dev blocking recall + oracle ceiling (+ training)
RUN_LOCKBOX = True         # honest accuracy on 45,000 held-out train S1
RUN_TEST    = True         # test blocking + scoring + submission files
CHECK_IDS   = False        # full validator id check (~20 min); do it for the final submission

REDO_PREP  = False         # True after changing normalization (then also change BLOCK_TAG)

## 2. Setup: code, packages, data, previous state

In [ ]:
import glob, os, shutil, subprocess, sys, time

WORK   = "/kaggle/temp/amz"                 # scratch disk (large, not saved)
SAVE   = "/kaggle/working"                  # saved as the notebook output (max ~20 GB)
VENV   = "/kaggle/temp/venv"
os.makedirs("/kaggle/temp", exist_ok=True)

def sh(cmd, check=True, env=None, cwd=None):
    """Run a shell command, stream its output, fail loudly."""
    print(f"$ {cmd}", flush=True)
    t = time.time()
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (WORK if os.path.isdir(WORK) else None),
                         env={**os.environ, **(env or {})}, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print(f"[exit {p.returncode}, {time.time() - t:.0f}s]", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

# --- code: clone (or update) the GitHub repo. Private repo: add a Kaggle secret GITHUB_TOKEN.
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
auth = ""
if token:
    import base64
    b = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = f'-c http.extraHeader="Authorization: Basic {b}"'   # never written to .git/config
if os.path.isdir(f"{WORK}/.git"):
    sh(f"git {auth} pull --ff-only origin {BRANCH}", cwd=WORK)
else:
    sh(f"git {auth} clone --branch {BRANCH} {REPO_URL} {WORK}", cwd="/kaggle/temp")
sh("git log -1 --oneline", cwd=WORK)

# --- packages: isolated venv with the pinned versions (does not touch Kaggle's own packages)
if not os.path.exists(f"{VENV}/bin/python"):
    sh(f"python -m venv {VENV}", cwd=WORK)
sh(f"{VENV}/bin/pip install -q --upgrade pip && {VENV}/bin/pip install -q -r requirements.txt")
PY = f"{VENV}/bin/python"
sh(f"{PY} --version")

# --- data: find the challenge TSVs among the attached inputs
hits = sorted(glob.glob("/kaggle/input/**/train/train_source1.tsv", recursive=True))
assert hits, "Challenge data not found: Add Input -> your private dataset with dataset/train and dataset/test"
DATA = os.path.dirname(os.path.dirname(hits[0]))
TRUTH = f"{DATA}/train/train_ground_truth.tsv"
assert os.path.exists(f"{DATA}/test/test_source1.tsv"), f"no test folder next to {DATA}/train"
print("DATA =", DATA)

# --- previous state: restore artifacts/ and output/ from an attached earlier run of this notebook
states = sorted(glob.glob("/kaggle/input/**/state.tar", recursive=True))
if states and not os.path.exists(f"{WORK}/artifacts/_done"):
    print("restoring", states[0])
    sh(f"tar -xf '{states[0]}' -C {WORK}")
elif not states:
    print("no previous state attached: starting fresh")

DONE = f"{WORK}/artifacts/_done"
os.makedirs(DONE, exist_ok=True)
ENV = {"POLARS_MAX_THREADS": str(os.cpu_count() or 4), "PYTHONUNBUFFERED": "1"}
print("CPUs", os.cpu_count())
sh("free -g; df -h /kaggle/temp /kaggle/working | cat")

### Helpers: skip finished steps, save state

In [ ]:
def step(name, cmd):
    """Run cmd once; a marker in artifacts/_done makes later runs skip it."""
    if os.path.exists(f"{DONE}/{name}"):
        print(f"skip {name} (done)")
        return
    log = f"{WORK}/logs/{name}.log"
    os.makedirs(os.path.dirname(log), exist_ok=True)
    sh(f"set -o pipefail; {cmd} 2>&1 | tee {log}", env=ENV)
    open(f"{DONE}/{name}", "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))

def save_state():
    """Pack artifacts/, output/, logs/ into /kaggle/working/state.tar (restored by the setup cell)."""
    os.makedirs(SAVE, exist_ok=True)
    keep = [d for d in ("artifacts", "output", "logs") if os.path.isdir(f"{WORK}/{d}")]
    sh(f"tar -cf {SAVE}/state.tar.tmp {' '.join(keep)} && mv {SAVE}/state.tar.tmp {SAVE}/state.tar")
    sh(f"du -sh {SAVE}/state.tar")

LB_CAND = f"artifacts/cand/train_lockbox_{BLOCK_TAG}"
DEV_CAND = f"artifacts/cand/train_dev_{BLOCK_TAG}"
TEST_CAND = f"artifacts/cand/test_{BLOCK_TAG}"
MODELS = f"artifacts/model/model_{MODEL_TAG}_*.txt"

def scored_view(cand):
    """Scores are cached per candidate-dir name, so give each model its own name (a symlink)."""
    view = f"{cand}__{MODEL_TAG}"
    if not os.path.lexists(f"{WORK}/{view}"):
        os.symlink(os.path.basename(cand), f"{WORK}/{view}")
    return view

## 3. Tests + normalization (prep)

In [ ]:
sh(f"{PY} -m pytest -q")
if REDO_PREP:
    for d in ("artifacts/norm", "artifacts/stats"):
        shutil.rmtree(f"{WORK}/{d}", ignore_errors=True)
    for f in glob.glob(f"{DONE}/prep_*"):
        os.remove(f)
step("prep_train", f"{PY} scripts/peakmem.py src.prep --data {DATA} --split train")
step("prep_test",  f"{PY} scripts/peakmem.py src.prep --data {DATA} --split test")
save_state()

## 4. Dev: blocking recall, oracle ceiling, training, out-of-fold F0.5

In [ ]:
DEV = "artifacts/splits/dev_s1.parquet"
if RUN_DEV or TRAIN_MODEL:
    step(f"block_dev_{BLOCK_TAG}", f"{PY} scripts/peakmem.py src.block --split train --s1-ids {DEV} "
                                   f"--tag train_dev_{BLOCK_TAG} --chunk 25000")
    step(f"recall_dev_{BLOCK_TAG}", f"{PY} -m src.eval.blocking_recall --cand {DEV_CAND} --s1-ids {DEV} --truth {TRUTH}")
    step(f"oracle_dev_{BLOCK_TAG}", f"{PY} -m src.eval.oracle --cand {DEV_CAND} --s1-ids {DEV} --truth {TRUTH}")
if TRAIN_MODEL:
    step(f"feat_dev_{BLOCK_TAG}", f"{PY} scripts/peakmem.py src.features --split train --cand {DEV_CAND} "
                                  f"--tag train_dev_{BLOCK_TAG} --truth {TRUTH}")
    step(f"train_{MODEL_TAG}", f"{PY} scripts/peakmem.py src.train --feat artifacts/feat/train_dev_{BLOCK_TAG}.parquet "
                               f"--s1-ids {DEV} --tag {MODEL_TAG} --truth {TRUTH}")
assert glob.glob(f"{WORK}/{MODELS}"), f"no models {MODELS}: set TRAIN_MODEL = True"
if os.path.exists(f"{WORK}/artifacts/model/oof_{MODEL_TAG}.parquet"):
    step(f"decide_oof_{MODEL_TAG}", f"{PY} -m src.eval.decide_oof --oof artifacts/model/oof_{MODEL_TAG}.parquet "
                                    f"--s1-ids {DEV} --truth {TRUTH}")
save_state()

## 5. Lockbox: the honest accuracy number

Use it at milestones only, and never tune on it. The table at the end is the main result: macro F0.5 for
ALL / US / India, a test-mix estimate, and the oracle ceiling.

In [ ]:
LB = "artifacts/splits/lockbox_s1.parquet"
if RUN_LOCKBOX:
    step(f"block_lockbox_{BLOCK_TAG}", f"{PY} scripts/peakmem.py src.block --split train --s1-ids {LB} "
                                       f"--tag train_lockbox_{BLOCK_TAG} --chunk 25000")
    view = scored_view(LB_CAND)
    out = f"output/lockbox_{BLOCK_TAG}__{MODEL_TAG}"
    step(f"predict_lockbox_{BLOCK_TAG}__{MODEL_TAG}",
         f"PYTHON={PY} PARTS=100 scripts/predict_loop.sh --split train --data {DATA} --cand {view} "
         f"--s1-ids {LB} --models '{MODELS}' --tau {TAU} --out {out}")
    oof = f"artifacts/model/oof_{MODEL_TAG}.parquet"
    oof_arg = f"--oof {oof} --dev-cand {DEV_CAND}" if os.path.exists(f"{WORK}/{oof}") and os.path.isdir(f"{WORK}/{DEV_CAND}") else ""
    step(f"eval_lockbox_{BLOCK_TAG}__{MODEL_TAG}",
         f"{PY} -m src.eval.lockbox --pred {out}/matching_results.tsv --cand {LB_CAND} {oof_arg} "
         f"--truth {TRUTH} --test-s1 {DATA}/test/test_source1.tsv --tau {TAU} "
         f"--out {out}/lockbox_report.md --tag '{BLOCK_TAG} / {MODEL_TAG} / tau {TAU}'")
    print(open(f"{WORK}/{out}/lockbox_report.md").read())
    save_state()

## 6. Test: blocking, scoring, submission files, validation

In [ ]:
if RUN_TEST:
    step(f"block_test_{BLOCK_TAG}", f"{PY} scripts/peakmem.py src.block --split test --tag test_{BLOCK_TAG} --chunk 25000")
    save_state()
    view = scored_view(TEST_CAND)
    out = f"output/test_{BLOCK_TAG}__{MODEL_TAG}"
    step(f"predict_test_{BLOCK_TAG}__{MODEL_TAG}",
         f"PYTHON={PY} PARTS=10 scripts/predict_loop.sh --split test --data {DATA} --cand {view} "
         f"--models '{MODELS}' --tau {TAU} --out {out}")
    ids = "--check-ids" if CHECK_IDS else ""
    step(f"validate_test_{BLOCK_TAG}__{MODEL_TAG}{'_ids' if CHECK_IDS else ''}",
         f"{PY} utils/validate_submission.py -m {out}/matching_results.tsv -c {out}/candidate_pairs.tsv "
         f"-t {DATA}/test {ids}")
    # submission files, easy to download from the Output tab
    dst = f"{SAVE}/submission_{BLOCK_TAG}__{MODEL_TAG}"
    os.makedirs(dst, exist_ok=True)
    for f in ("matching_results.tsv", "candidate_pairs.tsv", "prediction_stats.csv", "run_meta.json"):
        shutil.copy(f"{WORK}/{out}/{f}", dst)
    save_state()
    sh(f"ls -la {dst}")

## 7. Summary

In [ ]:
for f in sorted(glob.glob(f"{WORK}/logs/*.log")):
    if any(k in f for k in ("recall_dev", "oracle_dev", "decide_oof")):
        print(f"==== {os.path.basename(f)}"); print(open(f).read()[-3000:])
for f in sorted(glob.glob(f"{WORK}/output/*/lockbox_report.md")):
    print(f"==== {f}"); print(open(f).read()[-4000:])
sh(f"ls -la {SAVE}")